# Fraud Analytics Questions

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

## Retrieve Silver tables


In [0]:
%sql
USE CATALOG azure_databricks_jarvis;
USE SCHEMA silver;


In [0]:
fraud = spark.read.table("fraud_table")
mcc_codes = spark.read.table("mcc_codes_table")
users = spark.read.table("users_table")
transactions = spark.read.table("transactions_table")
cards = spark.read.table("cards_table")

## Which day(s) of the week sees the highest number of fraudulent transactions?

In [0]:

daily_window = Window.partitionBy("transaction_date")
gold_fraud_by_day = transactions \
    .join(fraud, transactions.id == fraud.transaction_id, "left") \
    .withColumn("value", F.coalesce(F.col("value"), F.lit(False))) \
    .withColumn("day_of_week", F.date_format("date","E"))\
    .withColumn("hour", F.hour("date")) \
    .withColumn("time_of_day",
        F.when((F.col("hour") >= 5) & (F.col("hour") < 12), "Morning")
        .when((F.col("hour") >= 12) & (F.col("hour") < 17), "Afternoon")
        .when((F.col("hour") >= 17) & (F.col("hour") < 21), "Evening")
        .otherwise("Night")  # 21:00–4:59
    ) \
    .withColumn("transaction_date", F.to_date(transactions.date)) \
    .withColumn("fraud_count", F.sum(fraud.value.cast("int")).over(daily_window)) \
    .withColumn("total_count", F.count("*").over(daily_window)) \

gold_fraud_by_day \
    .filter(F.col("value") == True) \
    .groupBy("day_of_week") \
    .agg(F.count("day_of_week").alias("count")) \
    .orderBy("count", ascending=False) \
    .show()

+-----------+-----+
|day_of_week|count|
+-----------+-----+
|        Sun| 2646|
|        Fri| 2284|
|        Thu| 2082|
|        Tue| 2037|
|        Mon| 1747|
|        Sat| 1434|
|        Wed| 1102|
+-----------+-----+



## What is the trend of the fraud rate (fraudulent transactions divided by total) over the past month?

In [0]:
max_date = gold_fraud_by_day.select(F.max("transaction_date")).collect()[0][0]
gold_fraud_by_day = gold_fraud_by_day.withColumn("fraud_rate", F.col("fraud_count") / F.col("total_count"))
gold_fraud_by_day \
    .filter(F.col("transaction_date") >= F.date_sub(F.lit(max_date), 30)) \
    .orderBy("transaction_date") \
    .select("transaction_date","day_of_week", F.format_number(F.col("fraud_rate")*100,2).alias("fraud_rate %") )\
    .dropDuplicates(["transaction_date"]) \
    .show(31)

+----------------+-----------+------------+
|transaction_date|day_of_week|fraud_rate %|
+----------------+-----------+------------+
|      2019-10-01|        Tue|        0.24|
|      2019-10-02|        Wed|        0.00|
|      2019-10-03|        Thu|        0.26|
|      2019-10-04|        Fri|        0.28|
|      2019-10-05|        Sat|        0.34|
|      2019-10-06|        Sun|        0.00|
|      2019-10-07|        Mon|        0.00|
|      2019-10-08|        Tue|        0.58|
|      2019-10-09|        Wed|        0.00|
|      2019-10-10|        Thu|        0.26|
|      2019-10-11|        Fri|        0.23|
|      2019-10-12|        Sat|        0.33|
|      2019-10-13|        Sun|        0.00|
|      2019-10-14|        Mon|        0.00|
|      2019-10-15|        Tue|        0.46|
|      2019-10-16|        Wed|        0.00|
|      2019-10-17|        Thu|        0.28|
|      2019-10-18|        Fri|        0.28|
|      2019-10-19|        Sat|        0.25|
|      2019-10-20|        Sun|  

## Which users have the largest number of flagged (`is_fraud = true`) transactions?

In [0]:
gold_fraud_by_user = transactions \
    .join(fraud, transactions.id == fraud.transaction_id, "inner") \
    .join(users, users.id == transactions.client_id, "inner") \
    .filter(fraud.value == True) \
    .groupby(users.id.alias("user_id")) \
    .agg(F.count(users.id).alias("num_fraud_transactions"))

gold_fraud_by_user\
    .orderBy("num_fraud_transactions", ascending=False)\
    .show()

+-------+----------------------+
|user_id|num_fraud_transactions|
+-------+----------------------+
|   1102|                    58|
|    209|                    52|
|     27|                    45|
|    155|                    44|
|   1128|                    43|
|    989|                    42|
|   1741|                    42|
|   1851|                    42|
|   1649|                    41|
|   1416|                    39|
|    408|                    39|
|   1926|                    39|
|   1571|                    39|
|    692|                    39|
|    359|                    39|
|   1992|                    38|
|   1725|                    38|
|   1341|                    37|
|   1569|                    37|
|    561|                    36|
+-------+----------------------+
only showing top 20 rows


## Are there any users showing a sharp rise in transaction amount compared to their weekly average?

In [0]:
gold_fraud_by_day = gold_fraud_by_day.withColumn("week", F.date_trunc("week", "date"))

weekly_avg = gold_fraud_by_day \
    .groupBy("client_id", "week") \
    .agg(F.avg("amount").alias("weekly_avg_amount"))

latest_week_per_user = weekly_avg \
    .groupBy("client_id") \
    .agg(F.max("week").alias("latest_week"))
weekly_avg_labeled = weekly_avg.join(latest_week_per_user, "client_id", "inner")

recent = weekly_avg_labeled \
    .filter(F.col("week") == F.col("latest_week")) \
    .select("client_id", F.col("weekly_avg_amount").alias("recent_week_avg"))

baseline = weekly_avg_labeled \
    .filter(F.col("week") != F.col("latest_week")) \
    .groupBy("client_id") \
    .agg(F.avg("weekly_avg_amount").alias("baseline_avg"))

spike_check = recent.join(baseline, "client_id", "inner") \
    .withColumn("pct_above_baseline", (F.col("recent_week_avg") - F.col("baseline_avg")) / F.col("baseline_avg"))

spike_check \
    .filter(F.col("pct_above_baseline") > 0.5) \
    .orderBy(F.col("pct_above_baseline").desc()) \
    .show(20)
spike_check.write.mode("overwrite").saveAsTable("gold.spike_check")

+---------+---------------+---------------+------------------+
|client_id|recent_week_avg|   baseline_avg|pct_above_baseline|
+---------+---------------+---------------+------------------+
|      313|   174.34500000|31.639159557322|      4.5104181792|
|      442|   235.94636364|48.066502211618|      3.9087483545|
|     1030|   116.51285714|28.805560587602|      3.0448043629|
|      220|   275.37000000|68.947747278950|      2.9938940845|
|     1781|   107.02000000|27.765577234094|      2.8544129336|
|     1871|   175.45600000|47.119226118538|      2.7236604769|
|     1616|   219.62375000|60.601398611209|      2.6240706491|
|      814|   111.82500000|33.009862213138|      2.3876239555|
|     1435|   140.31000000|41.715288645439|      2.3635150219|
|      628|   164.61900000|52.593071057290|      2.1300511016|
|      334|   173.43800000|56.282017939530|      2.0815881582|
|      920|    68.53571429|22.313461473314|      2.0714962971|
|     1815|   138.38000000|46.112638827797|      2.0009

## Which merchant categories exhibit the highest fraud rate?

In [0]:
gold_fraud_by_category = gold_fraud_by_day \
    .join(mcc_codes, gold_fraud_by_day.mcc == mcc_codes.code, "left") \
    .groupBy("mcc", "item") \
    .agg(
        F.sum(F.col("value").cast("int")).alias("fraud_count"),
        F.count("*").alias("total_count"),
        F.sum(F.when(F.col("value") == True, F.col("amount")).otherwise(0)).alias("total_fraud_amount")
    ) \
    .withColumn("fraud_rate", F.col("fraud_count") / F.col("total_count"))

gold_fraud_by_category\
    .filter(F.col("total_count") >= 30)\
    .orderBy(F.col("fraud_rate").desc())\
    .show(10)


+----+--------------------+-----------+-----------+------------------+--------------------+
| mcc|                item|fraud_count|total_count|total_fraud_amount|          fraud_rate|
+----+--------------------+-----------+-----------+------------------+--------------------+
|4411|        Cruise Lines|        165|        428|       185946.7800|  0.3855140186915888|
|5733|Music Stores - Mu...|         76|        319|         7562.2600| 0.23824451410658307|
|3006|Miscellaneous Fab...|         29|        351|         8543.3600| 0.08262108262108261|
|5045|Computers, Comput...|        204|       2793|        25615.1100| 0.07303974221267455|
|3144|Floor Covering St...|         23|        334|         6643.9200|  0.0688622754491018|
|5732|  Electronics Stores|        402|       6997|        61171.3800|  0.0574531942260969|
|3005|Miscellaneous Met...|         22|        391|         6943.7800|0.056265984654731455|
|3009|Fabricated Struct...|         22|        408|         6234.7400| 0.0539215

## Are there specific merchants with unusually high fraud volume?

In [0]:


gold_fraud_by_category\
    .filter(F.col("total_count") >= 30)\
    .orderBy(F.col("fraud_count").desc())\
    .select("mcc", "item", "fraud_count") \
    .show(10)

+----+--------------------+-----------+
| mcc|                item|fraud_count|
+----+--------------------+-----------+
|5311|   Department Stores|       2251|
|5300|     Wholesale Clubs|        991|
|5310|     Discount Stores|        859|
|4829|      Money Transfer|        725|
|5912|Drug Stores and P...|        479|
|5815|Digital Goods - M...|        449|
|5411|Grocery Stores, S...|        425|
|5732|  Electronics Stores|        402|
|5651|Family Clothing S...|        385|
|5719|Miscellaneous Hom...|        313|
+----+--------------------+-----------+
only showing top 10 rows


## How does fraud distribution vary by time of day (morning vs. night)?

In [0]:
gold_fraud_by_day \
    .groupBy("time_of_day") \
    .agg(
        F.sum(F.col("value").cast("int")).alias("fraud_count"),
        F.count("*").alias("total_count")
    ) \
    .withColumn("fraud_rate", F.col("fraud_count") / F.col("total_count")) \
    .select("time_of_day", F.format_number(F.col("fraud_rate")*100,2).alias("fraud rate %")) \
    .orderBy("fraud_rate", ascending=False) \
    .show()

+-----------+------------+
|time_of_day|fraud rate %|
+-----------+------------+
|  Afternoon|        0.13|
|    Morning|        0.11|
|    Evening|        0.08|
|      Night|        0.03|
+-----------+------------+



## What’s the average transaction amount for fraud vs. non-fraud transactions?

In [0]:

transactions_amount_FVNF = gold_fraud_by_day \
    .groupBy(gold_fraud_by_day.value) \
    .agg(
        F.avg("amount").alias("avg_amount"),
    ) \
    .select(F.col("value"), F.format_number("avg_amount",2).alias("avg_amount"))\
    .orderBy("value") 
transactions_amount_FVNF.write.mode("overwrite").saveAsTable("gold.transactions_amount_FVNF")

## Which merchant category has the highest total fraud amount?

In [0]:
gold_fraud_by_category\
    .orderBy(F.col("total_fraud_amount").desc())\
    .select("mcc", "item", "total_fraud_amount")\
    .show(1)

+----+-----------------+------------------+
| mcc|             item|total_fraud_amount|
+----+-----------------+------------------+
|5311|Department Stores|       225647.1900|
+----+-----------------+------------------+
only showing top 1 row


## What are the total monetary losses due to fraud each day?

In [0]:
monetary_losses_by_day = gold_fraud_by_day \
    .filter(F.col("value") == True) \
    .groupBy("transaction_date") \
    .agg(
        F.sum("amount").alias("total_fraud_loss")
    ) \
    .select("transaction_date", "total_fraud_loss") \
    .orderBy("transaction_date")
monetary_losses_by_day.write.mode("overwrite").saveAsTable("gold.monetary_losses_by_day")

## How many unique users commit fraudulent transactions per week?

In [0]:
gold_fraud_by_day = gold_fraud_by_day.withColumn("week", F.date_trunc('week', "date"))

gold_fraud_by_day \
    .filter(F.col("value") == True) \
    .groupBy("week") \
    .agg(F.countDistinct("client_id").alias("# of Unq users")) \
    .select("week", "# of Unq users") \
    .show(10)

+-------------------+--------------+
|               week|# of Unq users|
+-------------------+--------------+
|2012-04-02 00:00:00|            10|
|2018-04-30 00:00:00|            14|
|2018-06-04 00:00:00|            13|
|2013-02-25 00:00:00|             5|
|2013-10-21 00:00:00|            15|
|2013-11-18 00:00:00|             9|
|2018-05-14 00:00:00|             7|
|2012-02-13 00:00:00|             9|
|2010-05-10 00:00:00|            10|
|2019-06-03 00:00:00|             9|
+-------------------+--------------+
only showing top 10 rows


## Do fraud patterns show seasonal or monthly spikes?

In [0]:

gold_fraud_by_day = gold_fraud_by_day \
    .withColumn("month", F.month("date")) \
    .withColumn("season",
        F.when(F.col("month").isin(12, 1, 2), "Winter")
         .when(F.col("month").isin(3, 4, 5), "Spring")
         .when(F.col("month").isin(6, 7, 8), "Summer")
         .otherwise("Fall")
    )

# by month
gold_fraud_by_day \
    .groupBy("month") \
    .agg(
        F.sum(F.col("value").cast("int")).alias("fraud_count"),
        F.count("*").alias("total_count")
    ) \
    .withColumn("fraud_rate", F.col("fraud_count") / F.col("total_count")) \
    .orderBy("month") \
    .show(12)

# by season
gold_fraud_by_day \
    .groupBy("season") \
    .agg(
        F.sum(F.col("value").cast("int")).alias("fraud_count"),
        F.count("*").alias("total_count")
    ) \
    .withColumn("fraud_rate", F.col("fraud_count") / F.col("total_count")) \
    .orderBy(F.col("fraud_rate").desc()) \
    .show()

+-----+-----------+-----------+--------------------+
|month|fraud_count|total_count|          fraud_rate|
+-----+-----------+-----------+--------------------+
|    1|       1003|    1139155|8.804771958161971E-4|
|    2|       1030|    1031351|9.986900676879162E-4|
|    3|       1167|    1145390|0.001018866936152...|
|    4|       1171|    1106182|0.001058596144214...|
|    5|       1097|    1146194|9.570805640231933E-4|
|    6|        872|    1118522|7.796002224363937E-4|
|    7|       1118|    1153675|9.690770797668321E-4|
|    8|       1328|    1156873|0.001147922027742...|
|    9|       1081|    1117795|9.670825151302341E-4|
|   10|       1175|    1148638|0.001022950659824...|
|   11|       1089|    1003488|0.001085214770879...|
|   12|       1201|    1038652|0.001156306443351575|
+-----+-----------+-----------+--------------------+

+------+-----------+-----------+--------------------+
|season|fraud_count|total_count|          fraud_rate|
+------+-----------+-----------+-----------

In [0]:
monthly = gold_fraud_by_day \
    .withColumn("month", F.month("date")) \
    .groupBy("month") \
    .agg(
        F.sum(F.col("value").cast("int")).alias("fraud_count"),
        F.count("*").alias("total_count")
    ) \
    .withColumn("fraud_rate", F.col("fraud_count") / F.col("total_count"))

overall_avg = monthly.select(F.avg("fraud_rate")).first()[0]

monthly \
    .withColumn("pct_above_avg", (F.col("fraud_rate") - F.lit(overall_avg)) / F.lit(overall_avg)) \
    .select("month", "fraud_rate", "pct_above_avg") \
    .orderBy(F.col("pct_above_avg").desc()) \
    .show(12)

+-----+--------------------+--------------------+
|month|          fraud_rate|       pct_above_avg|
+-----+--------------------+--------------------+
|   12|0.001156306443351575| 0.15228643984893986|
|    8|0.001147922027742...| 0.14393117341546857|
|   11|0.001085214770879...| 0.08144192397934918|
|    4|0.001058596144214...| 0.05491583936836751|
|   10|0.001022950659824...| 0.01939427971552955|
|    3|0.001018866936152...|0.015324753476521326|
|    2|9.986900676879162E-4|-0.00478196828769...|
|    7|9.690770797668321E-4|-0.03429200398893632|
|    9|9.670825151302341E-4|-0.03627963434191815|
|    5|9.570805640231933E-4|-0.04624681276502673|
|    1|8.804771958161971E-4|-0.12258386246253405|
|    6|7.796002224363937E-4|-0.22311012795806184|
+-----+--------------------+--------------------+



## How has user behavior changed before versus after a fraudulent event?

In [0]:
# Get each user's first fraud event date
first_fraud = gold_fraud_by_day \
    .filter(F.col("value") == True) \
    .groupBy("client_id") \
    .agg(F.min("transaction_date").alias("fraud_date"))

# Join back to all transactions for those users
user_txns = gold_fraud_by_day.join(first_fraud, "client_id", "inner")

# Label each transaction as before/after their fraud event
user_behaviour = user_txns \
    .withColumn("days_from_fraud", F.datediff("transaction_date", "fraud_date")) \
    .withColumn("period",
        F.when((F.col("days_from_fraud") >= -30) & (F.col("days_from_fraud") < 0), "before")
         .when((F.col("days_from_fraud") > 0) & (F.col("days_from_fraud") <= 30), "after")
         .otherwise(None)
    ) \
    .filter(F.col("period").isNotNull())

# Compare average amount and transaction count, before vs after
Avg_Amount_BVA = user_behaviour \
    .groupBy("period") \
    .agg(
        F.round(F.avg("amount"), 2).alias("avg_amount"),
        F.count("*").alias("txn_count"),
        F.countDistinct("client_id").alias("num_users")
    ) 
Avg_Amount_BVA.show()
Avg_Amount_BVA.write.mode("overwrite").saveAsTable("gold.Avg_Amount_BVA")

+------+----------+---------+---------+
|period|avg_amount|txn_count|num_users|
+------+----------+---------+---------+
| after|     45.37|   117415|     1196|
|before|     43.14|   111622|     1195|
+------+----------+---------+---------+



## Are fraudulent transactions more common on high-value purchases compared to low-value purchases?

In [0]:
gold_fraud_by_day = gold_fraud_by_day \
    .withColumn("amount_bucket",
        F.when(F.col("amount") < 25, "Low ($0-25)")
         .when((F.col("amount") >= 25) & (F.col("amount") < 100), "Medium ($25-100)")
         .when((F.col("amount") >= 100) & (F.col("amount") < 500), "High ($100-500)")
         .otherwise("Very High ($500+)")
    )

gold_fraud_by_day \
    .groupBy("amount_bucket") \
    .agg(
        F.sum(F.col("value").cast("int")).alias("fraud_count"),
        F.count("*").alias("total_count")
    ) \
    .withColumn("fraud_rate", F.col("fraud_count") / F.col("total_count")) \
    .orderBy("fraud_rate", ascending=False) \
    .show()

+-----------------+-----------+-----------+--------------------+
|    amount_bucket|fraud_count|total_count|          fraud_rate|
+-----------------+-----------+-----------+--------------------+
|Very High ($500+)|        356|      43364|0.008209574762475786|
|  High ($100-500)|       4726|    1516248|0.003116904358653...|
| Medium ($25-100)|       4421|    5578205| 7.92548857562603E-4|
|      Low ($0-25)|       3829|    6168098|6.207748320470913E-4|
+-----------------+-----------+-----------+--------------------+



In [0]:
gold_fraud_by_day.write.mode("overwrite").saveAsTable("gold.gold_fraud_by_day")
gold_fraud_by_category.write.mode("overwrite").saveAsTable("gold.gold_fraud_by_category")
gold_fraud_by_user.write.mode("overwrite").saveAsTable("gold.gold_fraud_by_user")
user_behaviour.write.mode("overwrite").saveAsTable("gold.user_behaviour")